# Pi* — Entraînement Deep RL (RecurrentPPO + LSTM)

**Workflow :**
1. Installer les dépendances (CUDA auto-détecté)
2. Monter Google Drive (données + sauvegarde modèle)
3. Télécharger les données Binance BTCUSDT futures 1m
4. Entraîner RecurrentPPO avec LSTM
5. Sauvegarder `deep_agent.zip` → Drive → télécharger sur votre PC

**Runtime recommandé :** GPU T4 ou A100 (Runtime → Modifier le type de runtime)

## 1. Installation

In [ ]:
import subprocess, sys

# PyTorch est déjà installé sur Colab avec CUDA
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'stable-baselines3>=2.3.0',
                'sb3-contrib>=2.3.0',
                'gymnasium>=0.29.0',
                'pyarrow'], check=True)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

## 2. Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/pi_star')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Dossier Drive : {DRIVE_DIR}')

## 3. Données Binance BTCUSDT futures 1m

In [ ]:
import io, zipfile, requests
from datetime import date
import pandas as pd

PARQUET_PATH = DRIVE_DIR / 'btcusdt_1m.parquet'

_BASE_URL = 'https://data.binance.vision/data/futures/um/monthly/klines'
_SYMBOL   = 'BTCUSDT'
_INTERVAL = '1m'
_COLUMNS  = ['open_time','open','high','low','close','volume',
             'close_time','quote_vol','trades',
             'taker_buy_vol','taker_buy_quote_vol','ignore']

def download_binance(start=(2020,1), end=None):
    if PARQUET_PATH.exists():
        df = pd.read_parquet(PARQUET_PATH)
        print(f'Parquet existant chargé : {len(df):,} bougies')
        return df

    if end is None:
        today = date.today()
        end = (today.year, today.month-1) if today.month > 1 else (today.year-1, 12)

    frames, year, month = [], start[0], start[1]
    ey, em = end

    while (year, month) <= (ey, em):
        tag = f'{year}-{month:02d}'
        url = f'{_BASE_URL}/{_SYMBOL}/{_INTERVAL}/{_SYMBOL}-{_INTERVAL}-{tag}.zip'
        print(f'  {tag}...', end=' ', flush=True)
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            with zipfile.ZipFile(io.BytesIO(r.content)) as z:
                df_m = pd.read_csv(z.open(z.namelist()[0]), header=None, names=_COLUMNS)
            frames.append(df_m)
            print('OK')
        except Exception as e:
            print(f'ERREUR ({e})')
        month += 1
        if month > 12: month, year = 1, year+1

    df = pd.concat(frames, ignore_index=True)

    # Conversion numérique — élimine les lignes header résiduelles (NaN après coerce)
    for c in ['open_time','open','high','low','close','volume','taker_buy_vol']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.dropna(subset=['open_time','close'])   # supprime lignes non-numériques

    df['ts']        = df['open_time'].astype('int64') // 1000
    df['timestamp'] = pd.to_datetime(df['ts'], unit='s', utc=True)
    df['funding_rate']  = 0.0
    df['open_interest'] = 0.0
    df = (df[['ts','timestamp','open','high','low','close','volume',
              'taker_buy_vol','funding_rate','open_interest']]
          .drop_duplicates('ts')
          .sort_values('ts')
          .reset_index(drop=True))
    df.to_parquet(PARQUET_PATH, index=False)
    print(f'\n{len(df):,} bougies sauvegardées → {PARQUET_PATH}')
    return df

START = (2020, 1)
END   = None

df_1m = download_binance(start=START, end=END)
print(f'\nPériode : {df_1m["timestamp"].iloc[0]} → {df_1m["timestamp"].iloc[-1]}')

## 4. Feature Engineering + Environnement

In [ ]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces

# ── Features ────────────────────────────────────────────────────
N_FEATURES   = 9
_CLOSE_CLIP  = 0.03
_VOL_CLIP    = 5.0
_FUND_CLIP   = 0.005
FEATURE_COLS = ['f_close_pct','f_body_pct','f_upper_wick','f_lower_wick',
                'f_vol_ratio','f_delta','f_funding','f_session']

def aggregate_5m(df_1m):
    df = df_1m.copy()
    df['ts_5m'] = (df['ts'] // 300) * 300
    agg = df.groupby('ts_5m').agg(
        open=('open','first'), high=('high','max'),
        low=('low','min'),     close=('close','last'),
        volume=('volume','sum'),
        taker_buy_vol=('taker_buy_vol','sum'),
        funding_rate=('funding_rate','last'),
    ).reset_index().rename(columns={'ts_5m':'ts'})
    agg['timestamp'] = pd.to_datetime(agg['ts'], unit='s', utc=True)
    return agg.sort_values('ts').reset_index(drop=True)

def label_session(ts_utc):
    hour = ts_utc.dt.hour
    s = pd.Series(0, index=ts_utc.index, dtype=int)
    s[(hour >= 7)  & (hour < 13)] = 1
    s[(hour >= 13) & (hour < 16)] = 2
    s[(hour >= 16) & (hour < 22)] = 3
    return s

def compute_features(df_1m):
    df = aggregate_5m(df_1m)
    df['session'] = label_session(df['timestamp'])
    close, open_, high, low, vol = df['close'], df['open'], df['high'], df['low'], df['volume']
    rng = (high - low).clip(lower=1e-9)

    df['f_close_pct']  = (close.pct_change().fillna(0).clip(-_CLOSE_CLIP,_CLOSE_CLIP) / _CLOSE_CLIP).astype('float32')
    df['f_body_pct']   = ((close - open_) / rng).clip(-1, 1).astype('float32')
    df['f_upper_wick'] = ((high - pd.concat([open_,close],axis=1).max(axis=1)) / rng).clip(0,1).astype('float32')
    df['f_lower_wick'] = ((pd.concat([open_,close],axis=1).min(axis=1) - low) / rng).clip(0,1).astype('float32')
    avg_vol = vol.rolling(20, min_periods=1).mean().clip(lower=1e-9)
    df['f_vol_ratio']  = ((vol / avg_vol).clip(0,_VOL_CLIP) / _VOL_CLIP).astype('float32')
    # Delta : ratio volume acheteur réel (Binance taker_buy_vol)
    df['f_delta']      = (df['taker_buy_vol'] / vol.clip(lower=1e-9)).clip(0,1).astype('float32')
    df['f_funding']    = (df['funding_rate'].fillna(0).clip(-_FUND_CLIP,_FUND_CLIP) / _FUND_CLIP).astype('float32')
    df['f_session']    = (df['session'].astype(float) / 3.0).astype('float32')
    return df.reset_index(drop=True)

def obs_from_row(row, position):
    return np.concatenate([row[FEATURE_COLS].values.astype(np.float32),
                           np.array([float(position)], dtype=np.float32)])

# ── Environnement ────────────────────────────────────────────────
FEE, SLIP, PENALTY = 0.0005, 0.0002, 1.5
_SL_LOW, _TP_LOW   = 0.003, 0.006
_SL_HIGH, _TP_HIGH = 0.004, 0.008
_SL_EXT, _TP_EXT   = 0.006, 0.012

def sl_tp(vol_ratio):
    if vol_ratio > 0.8:  return _SL_EXT,  _TP_EXT
    if vol_ratio > 0.5:  return _SL_HIGH, _TP_HIGH
    return _SL_LOW, _TP_LOW

class DeepTradingEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self, df_feat):
        super().__init__()
        self._sessions = self._split(df_feat)
        self._idx = 0
        self.observation_space = spaces.Box(-1., 2., shape=(N_FEATURES,), dtype=np.float32)
        self.action_space = spaces.Discrete(3)
        self._ep = self._pos = self._step = 0
        self._price = self._sl = self._tp = None

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._ep   = self._sessions[self._idx].reset_index(drop=True)
        self._idx  = (self._idx + 1) % len(self._sessions)
        self._step = 0
        self._pos  = 0
        self._price = None
        self._sl, self._tp = _SL_LOW, _TP_LOW
        return self._obs(), {}

    def step(self, action):
        row   = self._ep.iloc[self._step]
        price = float(row['close'])
        pnl   = self._pnl(price)
        sl_hit = self._pos != 0 and (pnl <= -self._sl or pnl >= self._tp)
        if sl_hit: action = 0

        desired = {0:0, 1:1, 2:-1}[int(action)]
        closing = self._pos != 0 and (desired == 0 or desired != self._pos)
        reward  = (pnl * PENALTY if pnl < 0 else pnl) if (closing or sl_hit) else 0.0

        self._act(int(action), price, row)
        self._step += 1
        done = self._step >= len(self._ep)

        if done and self._pos != 0:
            lp  = float(self._ep.iloc[-1]['close'])
            lpl = self._pnl(lp)
            reward += lpl * PENALTY if lpl < 0 else lpl
            self._pos = 0; self._price = None

        obs = self._obs() if not done else np.zeros(N_FEATURES, dtype=np.float32)
        return obs, float(reward), done, False, {}

    def render(self): pass

    @property
    def n_episodes(self): return len(self._sessions)

    def _obs(self):
        return obs_from_row(self._ep.iloc[min(self._step, len(self._ep)-1)], self._pos)

    def _pnl(self, price):
        if not self._pos or self._price is None: return 0.0
        return float(self._pos * (price - self._price) / self._price - FEE - SLIP)

    def _act(self, action, price, row):
        d = {0:0, 1:1, 2:-1}[action]
        if d == self._pos: return
        if self._pos: self._pos = 0; self._price = None
        if d:
            self._pos   = d
            self._price = price * (1 + d * SLIP)
            self._sl, self._tp = sl_tp(float(row.get('f_vol_ratio', 0.)))

    def _split(self, df):
        df = df.copy()
        df['_k'] = df['timestamp'].dt.date.astype(str) + '_' + df['session'].astype(str)
        return [g.drop(columns=['_k']) for _, g in df.groupby('_k', sort=True) if len(g) >= 6]

# Calcul des features
print('Calcul des features 5min...')
df_feat = compute_features(df_1m)
print(f'  {len(df_feat):,} bougies 5min | {len([c for c in df_feat.columns if c.startswith("f_")])} features')

# Test rapide de l'environnement
env_test = DeepTradingEnv(df_feat)
obs, _ = env_test.reset()
print(f'  obs shape : {obs.shape} — OK')
print(f'  Sessions  : {env_test.n_episodes}')

## 5. Entraînement RecurrentPPO (LSTM)

In [ ]:
from sb3_contrib import RecurrentPPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.callbacks import CheckpointCallback
import torch

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
TIMESTEPS  = 3_000_000   # ~3M = bon compromis qualité/temps sur T4
MODEL_PATH = str(DRIVE_DIR / 'deep_agent')
NORM_PATH  = str(DRIVE_DIR / 'deep_vecnorm.pkl')
CKPT_DIR   = str(DRIVE_DIR / 'checkpoints')

print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

# Vectorized env avec normalisation des observations et rewards
vec_env = make_vec_env(lambda: DeepTradingEnv(df_feat), n_envs=1)
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True,
                       clip_obs=10.0, gamma=0.99)

# Sauvegarde automatique toutes les 100k steps
checkpoint_cb = CheckpointCallback(
    save_freq       = 100_000,
    save_path       = CKPT_DIR,
    name_prefix     = 'deep_agent',
    save_vecnormalize = True,
)

model = RecurrentPPO(
    'MlpLstmPolicy',
    vec_env,
    verbose       = 1,
    device        = DEVICE,
    n_steps       = 2048,      # steps par rollout
    batch_size    = 64,
    n_epochs      = 10,
    learning_rate = 3e-4,
    gamma         = 0.99,
    gae_lambda    = 0.95,
    clip_range    = 0.2,
    ent_coef      = 0.01,      # encourage l'exploration
    policy_kwargs = {
        'lstm_hidden_size' : 64,
        'n_lstm_layers'    : 1,
        'net_arch'         : [64, 64],
    },
)

n_sess = DeepTradingEnv(df_feat).n_episodes
print(f'\nModèle RecurrentPPO (LSTM 64) — {TIMESTEPS:,} timesteps')
print(f'Sessions disponibles : {n_sess} | ~{TIMESTEPS // (n_sess * 50)} passes/session\n')

model.learn(total_timesteps=TIMESTEPS, callback=checkpoint_cb, progress_bar=True)

# Sauvegarde finale
model.save(MODEL_PATH)
vec_env.save(NORM_PATH)
print(f'\nModèle sauvegardé : {MODEL_PATH}.zip')
print(f'VecNormalize      : {NORM_PATH}')

## 6. Télécharger le modèle sur votre PC

In [ ]:
from google.colab import files

# Télécharge les 2 fichiers nécessaires pour l'inférence locale
files.download(MODEL_PATH + '.zip')   # le réseau LSTM
files.download(NORM_PATH)             # la normalisation des observations

print('Placez ces fichiers dans : C:/Users/PC/analyser/db/')
print('  → db/deep_agent.zip')
print('  → db/deep_vecnorm.pkl')

## 7. (Optionnel) Reprendre un entraînement existant

Si votre session Colab expire, relancez depuis la cellule 3 puis exécutez la cellule ci-dessous :

In [ ]:
# Reprendre depuis le dernier checkpoint
RESUME_MODEL = str(DRIVE_DIR / 'deep_agent')   # ou un checkpoint spécifique
RESUME_NORM  = str(DRIVE_DIR / 'deep_vecnorm.pkl')

vec_env_r = make_vec_env(lambda: DeepTradingEnv(df_feat), n_envs=1)
vec_env_r = VecNormalize.load(RESUME_NORM, vec_env_r)
vec_env_r.training = True

model_r = RecurrentPPO.load(RESUME_MODEL, env=vec_env_r, device=DEVICE)
print(f'Reprise depuis {RESUME_MODEL}')
print(f'Timesteps déjà effectués : {model_r.num_timesteps:,}')

model_r.learn(total_timesteps=1_000_000, callback=checkpoint_cb,
              reset_num_timesteps=False, progress_bar=True)
model_r.save(MODEL_PATH)
vec_env_r.save(NORM_PATH)